###

In [2]:
import os
from dotenv import load_dotenv
from graphdatascience import GraphDataScience
import pandas as pd

In [ ]:
#Storing Neo4j connection details in variables
URI = "neo4j://127.0.0.1:7687"
USER = "neo4j"
PASSWORD = ""
DB_NAME = ""

In [4]:
gds = GraphDataScience(
    URI,
    auth=(USER, PASSWORD),
    database=DB_NAME
)
print("Connected to Neo4j GDS server version:", gds.version())

Connected to Neo4j GDS server version: 2.25.0


In [5]:
dense_wcc_id = 2 

In [6]:
def drop_if_exists(graph_name: str):
    graphs = gds.graph.list()
    if "graphName" in graphs.columns and graph_name in graphs["graphName"].values:
        gds.graph.drop(graph_name)

In [10]:
if gds.graph.exists("foodweb_directed").exists:
    gds.graph.drop("foodweb_directed")

In [11]:
if gds.graph.exists("foodweb_undirected").exists:
    gds.graph.drop("foodweb_undirected")

In [12]:
G_und, proj_result = gds.graph.project(
    "foodweb_undirected",
    {
        "Species": {
            "properties": []
        }
    },
    {
        "eaten_by": {
            "orientation": "UNDIRECTED",   
            "properties": []
        }
    }
)

In [13]:
dense = gds.wcc.write(
    G_und, 
    writeProperty="wccId"
)
print("WCC IDs written for predator to prey graph as 'wccId' property.\n")

WCC IDs written for predator to prey graph as 'wccId' property.



In [14]:
# 1) Dense: predator -> prey  (reverse edges for "who points to prey" view)
drop_if_exists("dense_pred_to_prey")
G_dense_pred_to_prey, _ = gds.graph.project.cypher(
    "dense_pred_to_prey",
    f"""
    MATCH (s:Species)
    WHERE s.wccId = {dense_wcc_id}
    RETURN id(s) AS id
    """,
    f"""
    MATCH (prey:Species)-[:eaten_by]->(pred:Species)
    WHERE prey.wccId = {dense_wcc_id} AND pred.wccId = {dense_wcc_id}
    RETURN id(pred) AS source, id(prey) AS target
    """
)
print("Dense projections created: - dense_pred_to_prey")

Dense projections created: - dense_pred_to_prey


In [15]:
# 2) Dense: prey -> predator  (:eaten_by means prey -> predator)
drop_if_exists("dense_prey_to_pred")
G_dense_prey_to_pred, _ = gds.graph.project.cypher(
    "dense_prey_to_pred",
    f"""
    MATCH (s:Species)
    WHERE s.wccId = {dense_wcc_id}
    RETURN id(s) AS id
    """,
    f"""
    MATCH (prey:Species)-[:eaten_by]->(pred:Species)
    WHERE prey.wccId = {dense_wcc_id} AND pred.wccId = {dense_wcc_id}
    RETURN id(prey) AS source, id(pred) AS target
    """
)
print("Dense projections created: - dense_prey_to_pred")

Dense projections created: - dense_prey_to_pred


In [16]:
# 3) Dense UNDIRECTED (strict): add reverse edges so it behaves undirected
drop_if_exists("dense_undirected")
G_dense_undirected, _ = gds.graph.project.cypher(
    "dense_undirected",
    f"""
    MATCH (s:Species)
    WHERE s.wccId = {dense_wcc_id}
    RETURN id(s) AS id
    """,
    f"""
    MATCH (a:Species)-[:eaten_by]->(b:Species)
    WHERE a.wccId = {dense_wcc_id} AND b.wccId = {dense_wcc_id}
    RETURN id(a) AS source, id(b) AS target
    UNION
    MATCH (a:Species)-[:eaten_by]->(b:Species)
    WHERE a.wccId = {dense_wcc_id} AND b.wccId = {dense_wcc_id}
    RETURN id(b) AS source, id(a) AS target
    """
)
print("Dense projections created: - dense_undirected")

Dense projections created: - dense_undirected


### Finding primary consumers using reverse projection

In [18]:
query = """// Step 1: Identify producers
CALL gds.degree.stream(
  'dense_pred_to_prey',
  { orientation: 'NATURAL' }
)
YIELD nodeId, score AS indegree
WHERE indegree = 0
WITH gds.util.asNode(nodeId) AS producer
WHERE producer.taxon_kingdom IN ["Plantae", "Chromista"]

// Step 2: Find primary consumers
MATCH (producer)-[:eaten_by]->(primary:Species)

// Step 3: Return
RETURN DISTINCT primary.scientific_name AS scientific_name,
       primary.common_name AS common_name,
       primary.taxon_class AS class,
       primary.taxon_phylum AS phylum
ORDER BY scientific_name;"""

result = gds.run_cypher(query)
print(result)

               scientific_name                  common_name      class  \
0              Aceria lantanae      Lantana Flower Gallmite  Arachnida   
1             Aceria theospyri  persimmon leaf blister gall  Arachnida   
2           Adejeania vexatrix                         None    Insecta   
3          Aerophilus nigripes                         None    Insecta   
4                  Afrogegenes                      Dodgers    Insecta   
..                         ...                          ...        ...   
216          Zonocerus elegans                         None    Insecta   
217  Zonocerus elegans elegans                         None    Insecta   
218           Zosterops virens         Green Cape White-eye       Aves   
219    Zosterops virens virens         Green Cape White-eye       Aves   
220       Zygaena filipendulae              Six-spot Burnet    Insecta   

         phylum  
0    Arthropoda  
1    Arthropoda  
2    Arthropoda  
3    Arthropoda  
4    Arthropoda  
.. 

### Finding primary consumers using original projection

In [19]:
query = """CALL gds.degree.stream(
  'dense_prey_to_pred',
{ orientation: 'REVERSE' })
YIELD nodeId, score AS indegree
WHERE indegree = 0 
WITH gds.util.asNode(nodeId) AS producer
WHERE producer.taxon_kingdom in ["Plantae", "Chromista"]

// Step 2: Find primary consumers
MATCH (producer)-[:eaten_by]->(primary:Species)

RETURN primary.scientific_name as scientific_name,
        primary.common_name as common_name,
        primary.taxon_class as class,
        primary.taxon_phylum as phylum
ORDER BY producer.scientific_name;"""

result = gds.run_cypher(query)
print(result)

                           scientific_name  \
0                  Tamiasciurus hudsonicus   
1                        Trichodes ornatus   
2                         Vanessa atalanta   
3                   Strangalia luteicornis   
4                      Toxomerus geminatus   
..                                     ...   
431                Coniodictyum chevalieri   
432                        Giraffa giraffa   
433                Giraffa giraffa giraffa   
434               Tragelaphus strepsiceros   
435  Tragelaphus strepsiceros strepsiceros   

                              common_name              class         phylum  
0                   American Red Squirrel           Mammalia       Chordata  
1                 Ornate Checkered Beetle            Insecta     Arthropoda  
2                             Red Admiral            Insecta     Arthropoda  
3    Yellow-horned Flower Longhorn Beetle            Insecta     Arthropoda  
4                    Eastern Calligrapher            Inse